# 04 — A tool delivered through MCP
MCP changes how the tool is connected. CrewAI still gives the agent a tool.
The local server exposes only a read-only annual ROI calculation.



**Basic lesson 4 of 13.**


## Problem

A local tool and a separately hosted capability need different connection lifecycles.


## Concept

MCP supplies a standard tool connection; CrewAI still presents a tool to the agent.

MCP is a transport and discovery boundary, not a new reasoning model. A separate server publishes a tool; CrewAI's adapter converts the discovered capability into a tool an Agent can use.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process   ◀ YOU ARE HERE
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
Crew Task ──▶ Agent ──▶ MCPServerAdapter
                              │ starts stdio subprocess
                              ▼
                        FastMCP server ──▶ annual_roi
                              │ JSON tool result
                              └───────────────▶ Agent answer
```

CrewAI adapter ↔ stdio MCP server → annual_roi function. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Configure a local stdio server process

Configure a local stdio server process; this file is the client side.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
sys.path.insert(0, str(ROOT))

import workshop  # configure project-local CrewAI storage
from crewai import Agent, Crew, Process, Task
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters

from workshop.common import model

server = StdioServerParameters(
    command=sys.executable,
    args=[str(ROOT / "workshop" / "mcp_server.py")],
)



## Connect, inspect, and use the remote tool
The context manager starts and stops the stdio server. Never put a normal
print statement on the server's stdout; that channel carries MCP messages.



## Step 2 — Discover the tool inside the adapter's connection lifetime, then attach it to the CrewAI agent

Discover the tool inside the adapter's connection lifetime, then attach it to the CrewAI agent.


In [ ]:
with MCPServerAdapter(server) as tools:
    print("MCP tools:", [item.name for item in tools])
    analyst = Agent(
        role="ROI Analyst",
        goal="Calculate a planning scenario through MCP",
        backstory="You distinguish estimates from measured outcomes.",
        llm=model(),
        tools=tools,
        max_iter=3,
        allow_delegation=False,
    )
    task = Task(
        description="Use the MCP annual_roi tool with benefit 45000 and cost 30000.",
        expected_output="Scenario ROI percentage with a caveat.",
        agent=analyst,
    )
    print(Crew(agents=[analyst], tasks=[task], process=Process.sequential).kickoff().raw)



## Bonus scenario — a party planner
The same server now offers a second, unrelated tool. The agent picks the
right one from the descriptions alone, which is why tool descriptions matter.



In [ ]:
with MCPServerAdapter(server) as tools:
    print("MCP tools:", [item.name for item in tools])
    planner = Agent(
        role="Party Planner",
        goal="Order the right amount of food",
        backstory="You never run out of pizza.",
        llm=model(),
        tools=tools,
        max_iter=3,
        allow_delegation=False,
    )
    order = Task(
        description="We are 14 people and each eats 3 slices. How many pizzas do we order?",
        expected_output="The number of pizzas, from the tool.",
        agent=planner,
    )
    print(Crew(agents=[planner], tasks=[order], process=Process.sequential).kickoff().raw)


## Failure mode to know about

The adapter's context manager owns the server connection. A stray server `print()` corrupts stdio protocol messages; a missing tool name means discovery failed before the LLM ran.


## Production implication

Treat an MCP server as a separate trust boundary: pin the server, inspect discovered tools, constrain filesystem/network access, and close connections reliably.


## Your turn

List the server's tools, then compare their names with the local tool lesson.

**Checkpoint:** The adapter discovers annual_roi and closes its subprocess after use.
